In [0]:
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col,
    count,
    avg,
    date_format,
    to_date,
    lit,
    current_timestamp
)

spark.sql("CREATE SCHEMA IF NOT EXISTS citibike_lakehouse.gold")

process_timestamp = datetime.now(timezone.utc)

source_table = "citibike_lakehouse.silver.trips"

hourly_demand_table = "citibike_lakehouse.gold.hourly_demand"
daily_demand_table = "citibike_lakehouse.gold.daily_demand"
station_popularity_table = "citibike_lakehouse.gold.station_popularity"

df_trips = spark.table(source_table)

display(df_trips.limit(10))
df_trips.printSchema()

df_hourly_demand = (
    df_trips
    .groupBy("hour", "day_of_week")
    .agg(
        count("*").alias("trip_count"),
        avg("ride_duration_minutes").alias("avg_ride_duration_minutes")
    )
    .withColumn("_processed_at", lit(process_timestamp))
    .withColumn("_inserted_at", current_timestamp())
)

(
    df_hourly_demand.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(hourly_demand_table)
)

df_daily_demand = (
    df_trips
    .withColumn("ride_date", to_date(col("started_at")))
    .groupBy("ride_date")
    .agg(
        count("*").alias("trip_count"),
        avg("ride_duration_minutes").alias("avg_ride_duration_minutes")
    )
    .withColumn("_processed_at", lit(process_timestamp))
    .withColumn("_inserted_at", current_timestamp())
)

(
    df_daily_demand.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(daily_demand_table)
)

df_station_popularity = (
    df_trips
    .groupBy("start_station_name")
    .agg(
        count("*").alias("trip_count"),
        avg("ride_duration_minutes").alias("avg_ride_duration_minutes")
    )
    .filter(col("start_station_name").isNotNull())
    .orderBy(col("trip_count").desc())
    .withColumn("_processed_at", lit(process_timestamp))
    .withColumn("_inserted_at", current_timestamp())
)

(
    df_station_popularity.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(station_popularity_table)
)

In [0]:
%sql
SELECT * FROM citibike_lakehouse.gold.hourly_demand;

SELECT * FROM citibike_lakehouse.gold.daily_demand;

SELECT * FROM citibike_lakehouse.gold.station_popularity;